# 06 — Pré-processamento e normalização de treino

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))
print('Project root:', ROOT)

Project root: /Users/luizgabriel/Documents/ChatGPT/article


In [2]:
import json
from dataclasses import asdict
from ecg_ai.config import load_config
from ecg_ai.contracts import SignalSpec
from ecg_ai.data_loader import load_ptbxl_metadata
from ecg_ai.preprocessing import fit_training_normalization
metadata, _ = load_ptbxl_metadata(ROOT / 'data/raw/ptb-xl')
train = metadata.query('strat_fold <= 8').copy()
train['ecg_path'] = train.filename_lr.map(lambda value: str(ROOT / 'data/raw/ptb-xl' / value))
spec = fit_training_normalization(train, SignalSpec.from_dict(load_config(ROOT / 'configs/default.json')['signal']))
target = ROOT / 'data/processed/manifests/preprocessing.json'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(json.dumps(asdict(spec), indent=2))
asdict(spec)

{'lead_order': ('I',
  'II',
  'III',
  'aVR',
  'aVL',
  'aVF',
  'V1',
  'V2',
  'V3',
  'V4',
  'V5',
  'V6'),
 'target_fs': 100,
 'duration_seconds': 10.0,
 'bandpass_low_hz': 0.5,
 'bandpass_high_hz': 40.0,
 'clip_millivolts': 5.0,
 'normalization_mean': (-7.761769189943861e-05,
  -8.543861145629404e-05,
  -7.745649715332187e-06,
  8.165961429809712e-05,
  -3.185525205176407e-05,
  -4.3471591547791116e-05,
  5.058452661413963e-05,
  -1.7948831491268343e-05,
  -3.556041376855983e-05,
  -6.904003932724813e-05,
  -9.822538589332718e-05,
  -0.00017568742562333207),
 'normalization_std': (0.1482272406061903,
  0.1515659693601935,
  0.14712005708547893,
  0.13063893616079436,
  0.1267755338022293,
  0.1296379929654993,
  0.2074000202502824,
  0.3228245764515728,
  0.3139827637143342,
  0.28216256244937715,
  0.24518784034179736,
  0.19665665187695996)}

Os parâmetros são estimados somente nos folds oficiais 1–8 e reutilizados em validação, teste e inferência.